# talkinghead — text to video

Paste a script into cell 4, run everything, download the video.

## Before you run

1. **Settings → Accelerator → GPU**, **Internet → On**
2. **+ Add Input** → your private `talkinghead-assets` dataset
3. **+ Add Input** → `latentsync-weights` if you have built it. Optional, but it
   removes 8–12 minutes from every run. See the last cell.

## Where the time goes

| Stage | Time | Notes |
|---|---|---|
| pip installs | 4–6 min | unavoidable; sessions are ephemeral |
| weight download (9.8 GB) | 8–12 min | **skipped entirely when cached** |
| TTS | roughly real-time on GPU | per minute of speech |
| lipsync @ 256 | ~3–5 s per 1 s of video | |
| lipsync @ 512 | ~13.5 s per 1 s of video | measured in Phase 0 |

A 2-minute video at 256 with cached weights is roughly 8–10 minutes, against
~30 minutes cold at 512.

## Which profile

**Start with 720p / 256.** It is about 3× faster per frame, it fits comfortably,
and it uses the upstream `num_frames=16` — the configuration most likely to sync
correctly. 512 only fits at `num_frames=8`, which is the leading suspect for weak
mouth movement. `diagnose_lipsync.ipynb` settles that question.

## 1. Install

In [ ]:
REPO = "https://github.com/Deveshkumar742/Talkinghead.git"
PROFILE = "720p"          # "720p" (recommended) or "1080p"
INFERENCE_STEPS = 20      # lower is faster; time scales roughly linearly

import os, subprocess, sys, time
from pathlib import Path

PKG_DIR = Path("/kaggle/working/talkinghead")
if not PKG_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(PKG_DIR)], check=True)
os.chdir(PKG_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[lipsync]"],
               check=True)

os.environ["TH_PROFILE"] = PROFILE
# Module invocation, not the console script: pip's bin directory is not reliably
# on PATH in a notebook session.
TH = f"{sys.executable} -m talkinghead.cli"
print("installed, profile =", PROFILE)

In [ ]:
# Chatterbox for the voice.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chatterbox-tts"],
               check=False)

# LatentSync's own dependency list, installed one package at a time. A single
# unresolvable pin aborts the whole pip command and silently takes the rest with
# it -- which is exactly what went wrong twice during Phase 0.
PACKAGES = [
    "diffusers==0.32.2", "transformers==4.48.0", "decord==0.6.0", "accelerate",
    "einops", "omegaconf", "opencv-python", "mediapipe", "python_speech_features",
    "librosa", "scenedetect", "ffmpeg-python", "imageio", "imageio-ffmpeg",
    "lpips", "face-alignment", "kornia", "insightface==0.7.3", "onnxruntime-gpu",
    "DeepCache==0.1.1", "soundfile",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cython"],
               capture_output=True)
failed = [
    pkg for pkg in PACKAGES
    if subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                      capture_output=True).returncode != 0
]
print("failed:", failed or "none")

## 2. Check the environment and your assets

`check` validates the mounted recordings against the active profile — including
whether your base loop's resolution can support the output you asked for. If
something is wrong it names the file and the fix.

In [ ]:
!{TH} host
print("-" * 68)
!{TH} check || true

## 3. Find the cached weights

If `latentsync-weights` is attached this locates it and the render skips the
download entirely. Otherwise the provider fetches ~9.8 GB, which is most of the
wait.

In [ ]:
LATENTSYNC_DIR = Path("/kaggle/working/LatentSync")
CACHE_DIR = None

# A tarred dataset extracts once per session; an unpacked one is used in place.
for tar in Path("/kaggle/input").rglob("checkpoints.tar"):
    target = Path("/kaggle/working/ckpt_cache")
    if not (target / "checkpoints" / "latentsync_unet.pt").is_file():
        target.mkdir(exist_ok=True)
        print(f"extracting {tar} ...")
        subprocess.run(["tar", "-xf", str(tar), "-C", str(target)], check=True)
    CACHE_DIR = target / "checkpoints"
    break
else:
    for unet in Path("/kaggle/input").rglob("latentsync_unet.pt"):
        CACHE_DIR = unet.parent
        break

print("weight cache:", CACHE_DIR or "none — will download ~9.8 GB")
CACHE_ARG = f"--cache-dir {CACHE_DIR}" if CACHE_DIR else ""

## 4. Your script

Spoken verbatim — nothing rewrites or polishes it. Blank lines become longer
pauses, which is what makes multi-paragraph delivery sound deliberate rather than
breathless.

In [ ]:
SCRIPT = """\
Hi, I'm Devesh. This is the first end-to-end test of the pipeline.

I'm checking three things. Whether the mouth tracks the words, especially on
plosives like p and b. Whether the voice actually sounds like me. And whether the
whole thing holds together well enough to put in front of a client.

If it does, we scale it up.
"""

script_path = Path("/kaggle/working/script.txt")
script_path.write_text(SCRIPT, encoding="utf-8")

# Segmentation and a runtime estimate, before spending anything.
!{TH} prep {script_path}

## 5. Voice

Run this alone while iterating on wording — it is the cheap half. Each segment is
cached by a hash of its own text, so editing one sentence re-synthesizes one
sentence.

**Listen before continuing.** If the voice is wrong, no amount of lipsync fixes
it, and video is where the GPU time goes.

In [ ]:
VOICE = Path("/kaggle/working/out/voice.wav")
t0 = time.time()
!{TH} tts {script_path} -o {VOICE} -v
print(f"\nTTS took {time.time() - t0:.0f}s")

from IPython.display import Audio, display
if VOICE.exists():
    display(Audio(str(VOICE)))

## 6. Video

Fits the base loop to the narration length, repaints the mouth, then muxes and
encodes. Only this stage needs the GPU. Reuses the cached voice from cell 5.

In [ ]:
OUT = Path("/kaggle/working/out/video.mp4")
t0 = time.time()
!{TH} gen {script_path} -o {OUT} --repo-dir {LATENTSYNC_DIR} {CACHE_ARG} --steps {INFERENCE_STEPS} -v
print(f"\ntotal {time.time() - t0:.0f}s")

## 7. Watch it

Full frame, then the mouth enlarged — at full size a subtle articulation failure
is invisible.

Does the jaw drop on open vowels? Do the lips close on `m`, `b`, `p`? A mouth that
only blurs or twitches means the sync is wrong, not the sharpness.

In [ ]:
from IPython.display import Video, display

if OUT.exists():
    !ffprobe -v error -show_entries stream=width,height,duration -of default=noprint_wrappers=1 "{OUT}"
    display(Video(str(OUT), embed=True, width=760))

    ZOOM = OUT.with_name("video_mouth.mp4")
    subprocess.run([
        "ffmpeg", "-y", "-loglevel", "error", "-i", str(OUT),
        "-vf", "crop=iw/2:ih/3:iw/4:ih/2,scale=560:-2", str(ZOOM),
    ], check=True)
    print("mouth region:")
    display(Video(str(ZOOM), embed=True, width=560))
else:
    print("No output — read the error above.")

## 8. Make the next run faster

Run this **once**, download `checkpoints.tar` from the Output panel, and upload it
as a private dataset named `latentsync-weights`. Attach it in future sessions and
cell 3 finds it automatically.

That removes 8–12 minutes from every subsequent render — by far the largest
saving. After that, in order:

1. **`INFERENCE_STEPS` 20 → 10** in cell 1. Roughly halves lipsync time. Upstream
   documents 20–50, so 10 is below the sanctioned range — judge the output rather
   than assuming it is fine.
2. **Stay on the 256 crop.** About 3× faster per frame than 512, and more likely
   to sync correctly.
3. **Reuse the cache.** Re-running the same script skips completed stages;
   `work/<script-hash>/` survives within a session.

In [ ]:
CKPT = LATENTSYNC_DIR / "checkpoints"
TAR = Path("/kaggle/working/checkpoints.tar")

if CACHE_DIR is not None:
    print("Already using a cache — nothing to do.")
elif not (CKPT / "latentsync_unet.pt").is_file():
    print("No weights downloaded yet; run cell 6 first.")
elif TAR.exists():
    print(f"{TAR} already built ({TAR.stat().st_size / 1024**3:.1f} GB)")
else:
    # Uncompressed on purpose: these are already-compressed tensors, so gzip
    # would cost minutes and save almost nothing.
    resolved = CKPT.resolve()
    subprocess.run(["tar", "-cf", str(TAR), "-C", str(resolved.parent),
                    resolved.name], check=True)
    print(f"{TAR}  ({TAR.stat().st_size / 1024**3:.1f} GB)")
    print("\nDownload it, then create a private dataset 'latentsync-weights'.")